In [20]:
import pdfplumber
import pandas as pd
import re

In [27]:
def is_number(s):
    """Checks if a string looks like a financial number."""
    if not s: return False
    # Remove common financial formatting to check
    clean = str(s).replace(',', '').replace('(', '').replace(')', '').replace('-', '').strip()
    # Check if what's left is a digit
    return clean.replace('.', '', 1).isdigit()

def extract_and_fix_alignment(pdf_path, output_excel):
    all_rows = []

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            # Stream mode to find columns based on whitespace
            table_settings = {
                "vertical_strategy": "text", 
                "horizontal_strategy": "text",
                "snap_tolerance": 4,
            }
            
            tables = page.extract_tables(table_settings)
            
            for table in tables:
                for row in table:
                    # Clean whitespace from cells
                    cleaned_row = [str(cell).strip() if cell else "" for cell in row]
                    
                    # --- SMART ALIGNMENT LOGIC ---
                    # 1. Find the index where the DATA (numbers) starts
                    first_number_idx = -1
                    for i, cell in enumerate(cleaned_row):
                        if is_number(cell):
                            first_number_idx = i
                            break
                    
                    # 2. Rebuild the row
                    if first_number_idx > 0:
                        # Join everything BEFORE the first number as the Label
                        label_part = " ".join(cleaned_row[:first_number_idx])
                        # Everything AFTER (and including) the first number is Data
                        data_part = cleaned_row[first_number_idx:]
                        
                        # Create the fixed row
                        new_row = [label_part] + data_part
                        all_rows.append(new_row)
                    else:
                        # If no numbers found (like a header row), keep it as is
                        # We filter out empty lists
                        if any(cleaned_row):
                            all_rows.append(cleaned_row)

    # Convert to DataFrame
    # Note: We don't set columns yet because row lengths might still vary slightly
    df = pd.DataFrame(all_rows)

    # --- HEADER PROCESSING ---
    # Find the row with years (e.g., "2024", "Original") to use as header
    header_idx = -1
    for i, row in df.iterrows():
        # Join row to string to search for keywords
        row_str = " ".join([str(x) for x in row])
        if "2024" in row_str or "2018" in row_str:
            header_idx = i
            break
            
    if header_idx != -1:
        # Extract header row
        headers = list(df.iloc[header_idx])
        
        # Clean headers: remove None/NaN and strip whitespace
        headers = [str(h).strip() for h in headers if h]
        
        # Ensure the first header is "Line Item" or similar
        if is_number(headers[0]): 
            headers.insert(0, "Line Item")
        else:
            headers[0] = "Line Item" # Rename first col for clarity

        # Slice data to start after header
        df = df.iloc[header_idx+1:].reset_index(drop=True)
        
        # Adjust DataFrame width to match Header width
        # If data has more columns than headers, clip it. If less, add NaNs.
        df = df.iloc[:, :len(headers)] 
        df.columns = headers

    # --- FINAL NUMERIC CLEANUP ---
    def clean_currency(val):
        if not val: return None
        s = str(val).replace(',', '').replace(' ', '')
        if '(' in s and ')' in s:
            s = '-' + s.replace('(', '').replace(')', '')
        try:
            return float(s)
        except:
            return val

    # Apply to all columns except the first (Label)
    for col in df.columns[1:]:
        df[col] = df[col].apply(clean_currency)

    # Export
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)
    df = df.set_index('For the perio d ending')
    df.to_excel(output_excel)
    print(f"Successfully fixed alignment and saved to {output_excel}")
    return df

# Run it
df = extract_and_fix_alignment("AMZN EV.pdf", "Amazon_EV_Final.xlsx")
display(df.head())

Successfully fixed alignment and saved to Amazon_EV_Final.xlsx


,2018-12-31,2019-12-31,2020-12-31,2021-12-31,2022-12-31,2023-12-31,2024-12-31,2025-12-1,2025-12-31,2026-12-31
For the perio d ending,,,,,,,,,,
Market Capit alization,737467.27,920224.32,1638235.79,1697179.06,860328.0,1577593.02,2323998.27,2505359.02,None,None
- Cash & Eq uivalents,41250.0,55021.0,84396.0,96049.0,70026.0,86780.0,101202.0,94197.0,None,None
+ Preferred Equity,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,None
+ Minority I nterest,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,None
+ Total Deb t,49289.0,77533.0,100504.0,132318.0,154972.0,154556.0,147838.0,152737.0,None,None
